In [ ]:
# 1. Instal·lació de Chrome i Selenium (Colab Edition)
%%shell
wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google.list
apt-get update
apt-get install -y google-chrome-stable
pip install selenium==4.18.1 webdriver-manager
google-chrome-stable --version

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.keys import Keys
from webdriver_manager.chrome import ChromeDriverManager
from urllib.parse import quote
import glob
import os
import time
import shutil
import pandas as pd
from google.colab import files

In [ ]:
def web_driver():
    options = webdriver.ChromeOptions()
    options.add_argument('--no-sandbox')
    options.add_argument('--headless')
    options.add_argument('--disable-gpu')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument("--window-size=1920,1200")
    # Forcem la descàrrega a la carpeta content
    prefs = {"download.default_directory": "/content/"}
    options.add_experimental_option("prefs", prefs)
    
    service = Service(ChromeDriverManager().install())
    return webdriver.Chrome(service=service, options=options)

In [ ]:
def descarrega_xlsx(nou_nom):
    # Busca el fitxer brand_db_export descarregat
    xlsx_files = glob.glob('/content/brand_db_export*.xlsx')
    if not xlsx_files:
         xlsx_files = [f for f in glob.glob('/content/*.xlsx') if f not in ['/content/entitats.xlsx', '/content/log_results_v5.xlsx']]
    
    if not xlsx_files: return False

    xlsx_files.sort(key=os.path.getmtime, reverse=True)
    ultim_arxiu = xlsx_files[0]

    wipo_folder = '/content/wipo'
    if not os.path.exists(wipo_folder): os.makedirs(wipo_folder)
    
    safe_name = str(nou_nom).replace('/', '_').replace('\\', '_').replace(':', '_').replace('?', '_').replace('*', '_')
    nou_nom_complet = os.path.join(wipo_folder, safe_name + ".xlsx")
    
    try:
        if os.path.exists(nou_nom_complet): os.remove(nou_nom_complet)
        shutil.move(ultim_arxiu, nou_nom_complet)
        return True
    except Exception as e:
        print(f"      Error movent fitxer: {e}")
        return False

In [ ]:
def titular(driver, codi):
  print(f"Cercant: {codi}...")
  
  # 1. Anem a la pàgina principal i esperem la sessió
  driver.get("https://branddb.wipo.int/en/advancedsearch")
  
  try:
    # 2. ESPERA EXTRA-LLARGA per a sistemes lents en mode headless (60s)
    # Busquem el camp de busca amb una combinació de selecció més flexible
    wait_long = WebDriverWait(driver, 60)
    
    # Selector multi-objectiu: id exact de la WIPO, qualsevol input amb placeholder search, o el 1r text del formulari
    selector_multi = "//input[@id='anySearchText'] | //input[contains(@placeholder, 'Search')] | //input[@type='text']"
    
    try:
        search_input = wait_long.until(EC.element_to_be_clickable((By.XPATH, selector_multi)))
    except TimeoutException:
        # Si falla, guardem una captura de pantalla per veure l'ERROR de la pàgina
        driver.save_screenshot('debug_timeout_wipo.png')
        print(f"   ! Timeout al carregar la web per a '{codi}'. Revisa el fitxer debug_timeout_wipo.png!")
        return codi, "Timeout Pagina"
    
    # 3. TECLEIG SIMULAT del prefix de cerca
    search_input.clear()
    search_input.send_keys(f'applicant:"{codi}"')
    time.sleep(2) # Pausa tècnica
    search_input.send_keys(Keys.ENTER)

    # 4. Verificació de Resultats o Zero Resultats
    # De vegades la web diu 'Displaying 0-0' o simplement no treu resultats
    download_xpath = '//span[text()="Download results"]'
    
    wait_results = WebDriverWait(driver, 45)
    wait_results.until(lambda d: d.find_elements(By.XPATH, download_xpath) or 
                                "Displaying 0-0" in d.page_source or 
                                "No records found" in d.page_source)
    
    if not driver.find_elements(By.XPATH, download_xpath):
        print(f"   - Cap marca trobada per a {codi}")
        return codi, "No trobat"

    # 5. Descàrrega efectiva
    btn = driver.find_element(By.XPATH, download_xpath)
    driver.execute_script("arguments[0].scrollIntoView();", btn)
    time.sleep(1)
    btn.click()
    
    excel_opt = WebDriverWait(driver, 20).until(
        EC.element_to_be_clickable((By.XPATH, '//li//span[text()="EXCEL"]'))
    )
    excel_opt.click()

    # Pausa per assegurar la descàrrega en mode headless
    time.sleep(15)

    if descarrega_xlsx(codi):
        print(f"   [OK] Descarregat: {codi}")
        return codi, "1"
    else:
        print(f"   [?] Resultats detectats però el fitxer s'ha perdut per a {codi}")
        return codi, "Error fitxer"

  except Exception as e:
    # Guardar captura de pantalla de l'error per saber si hi ha un bloqueig o UI nova
    driver.save_screenshot(f'error_{codi.replace(" ", "_")}.png')
    print(f"   ! Error inesperat amb {codi}: {type(e).__name__}")
    return codi, f"Error {type(e).__name__}"

In [ ]:
# Començar el procés amb les entitats
if os.path.exists("entitats.xlsx"):
    df = pd.read_excel("entitats.xlsx")
    # Filtrar i netejar llista
    entities = df['Titular (entitat)'].apply(lambda x: str(x).rsplit(",", 1)[0].rsplit(' (', 1)[0].strip()).unique()

    log = []
    print(f"Iniciant driver (V5 - Ultra Robust Mode) per a {len(entities)} entitats...")
    driver = web_driver()
    
    try:
        for ent in entities:
            if ent and ent != 'nan':
                entitat, status = titular(driver, ent)
                log.append([entitat, status])
    finally:
        driver.quit()
        pd.DataFrame(log, columns=["Entitat", "Estat"]).to_excel("log_results_v5.xlsx", index=False)
        print("\nProcés finalitzat. Revisa log_results_v5.xlsx")
else:
    print("ATENCIÓ: No has pujat el fitxer 'entitats.xlsx' a la carpeta principal de Colab.")

In [ ]:
if os.path.exists('/content/wipo'):
    if os.path.exists('marques_wipo_export_v5.zip'): os.remove('marques_wipo_export_v5.zip')
    shutil.make_archive('marques_wipo_export_v5', 'zip', '/content/wipo')
    files.download('marques_wipo_export_v5.zip')
    print("ZIP de resultats generat correctament.")